# Example 12.2 / 1: Assign Arcs with Degree Equations

Companion notebook for **12.2 Routing Models** in the AMPL lab Documentation, section "Assign Arcs with Degree Equations". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Assign Arcs with Degree Equations

The solid arcs illustrate one connected tour. Degree equations can also admit the dotted disconnected cycle unless the model adds flow or subtour constraints.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES ordered;               # depot first, then customers
set ARCS within {NODES, NODES};  # allowed directed arcs
param cost {ARCS} >= 0;          # travel cost per arc
var Use {ARCS} binary;           # 1 if the tour uses arc (i,j)

minimize TravelCost: sum {(i,j) in ARCS} cost[i,j] * Use[i,j];  # total travel cost
subject to Leave {i in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one departure from every node
subject to Enter {j in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one arrival at every node

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {                      # row: from node, column: to node
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
selected = [arc for arc, used in ampl.var["Use"].to_dict().items() if used > 0.5]
print(ampl.solve_result, ampl.obj["TravelCost"].value(), selected)

solved 7.0 [('c1', 'c2'), ('c2', 'depot'), ('c3', 'c4'), ('c4', 'c3'), ('depot', 'c1')]


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same model; `-` in the `ARCS` table excludes self-arcs:

In [6]:
%%writefile data.dat
set NODES := depot c1 c2 c3 c4;
set ARCS:  depot c1 c2 c3 c4 :=
    depot    -    +  +  +  +
    c1       +    -  +  +  +
    c2       +    +  -  +  +
    c3       +    +  +  -  +
    c4       +    +  +  +  - ;
param cost:  depot c1 c2 c3 c4 :=
    depot    .    2  2  9  9
    c1       2    .  1  8  9
    c2       2    2  .  9  8
    c3       9    8  9  .  1
    c4       9    9  7  1  . ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
solve;
display solve_result, TravelCost;
display {(i,j) in ARCS: Use[i,j] > 0.5};

solve_result = solved
TravelCost = 7

set {(i,j) in ARCS: Use[i,j] > 0.5}  :=
(depot,c1)   (c1,c2)      (c2,depot)   (c3,c4)      (c4,c3);

